In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import numpy as np

In [3]:
from bayesgpt.simulators import NestedModelFamily
from bayesgpt.simulators.benchmarks import StandardDDM, CollapsingBoundDDM, SuperDDM

In [4]:
def sample_v(n, context=None):
    return np.random.normal(0.5, 0.1, n)

def sample_a(n, context=None):
    return np.random.uniform(1.0, 2.0, n)

def sample_angle(n, context=None):
    return np.random.uniform(0.2, 0.4, n)

def sample_s_z(n, context=None):
    return np.random.uniform(0.05, 0.15, n)

In [5]:
# Define global parameter space (superset)
param_names = [
    "v",  # drift
    "a",  # boundary
    "z",  # initial bias
    "tau",  # non-decision time
    "sigma",  # noise scale
    "angle",  # collapse rate
    "s_v",  # All 's' are variability
    "s_a",
    "s_z",
    "s_tau",
    "s_sigma",
    "s_angle",
]

In [6]:
model_family = NestedModelFamily(parameter_names=param_names)

In [7]:
# Define global fixed parameters (common across all variants)
global_fixed_parameters = {
    "z": 0.5,          # Starting point
    "sigma": 1.0,      # Noise standard deviation
    "s_tau": 0.0       # No variability in non-decision time
}

In [8]:
# Define variant configurations
variants = [
    # StandardDDM variant
    (
        "std_ddm_1",
        StandardDDM,
        {"v": sample_v, "a": sample_a},
        {
            "tau": 0.3,
            "s_v": 0.0,
            "angle": 0.0,
            "s_z": 0.0,
        },
        0.0
    ),
    # CollapsingBoundDDM variant
    (
        "coll_ddm_1",
        CollapsingBoundDDM,
        {"v": sample_v, "angle": sample_angle},
        {
            "a": 1.2,
            "tau": 0.3,
            "s_v": 0.0,
            "s_z": 0.0,
        },
        0.0
    ),
    # SuperDDM variant with v_components
    (
        "super_ddm_1",
        SuperDDM,
        {"s_z": sample_s_z},
        {
            "a": 1.8,
            "tau": 0.3,
            "s_v": 0.2,
            "angle": 0.0,
            "v_components": np.array([0.5, 1.0, 1.5]),
            "p_components": np.array([0.4, 0.3, 0.3]),
        },
        0.0
    ),
    # SuperDDM variant with v_schedule
    (
        "super_ddm_2",
        SuperDDM,
        {"angle": sample_angle},
        {
            "a": 2.0,
            "tau": 0.4,
            "s_v": 0.0,
            "s_z": 0.1,
            "v_schedule": np.array([0.7, 1.2]),
            "t_schedule": np.array([0.5, 0.5]),
        },
        0.0
    )
]

In [9]:
# Combine global and local fixed parameters for each variant
variants_with_global = [
    (name, model, free_params, {**global_fixed_parameters, **fixed_params}, fallback)
    for name, model, free_params, fixed_params, fallback in variants
]

In [10]:
# Add all variants at once using collect_all_variants
model_family.add_all_variants(variants_with_global)

In [11]:
batch_size = 100  # Adjust as needed
results = {}
for variant in model_family.variant_names:
    results[variant] = model_family.sample(variant, batch_size)

In [12]:
# Print shapes of all data
for variant, result in results.items():
    print(f"\nShapes for {variant}:")
    print("  sim_data:")
    for key, value in result['sim_data'].items():
        print(f"    {key}: {value.shape}")
    print(f"  full_params: {result['full_params'].shape}")
    print(f"  inference_conditions: {result['inference_conditions'].shape}")

# Display sample results
for variant, result in results.items():
    print(f"\nResults for {variant}:")
    print(f"  First 5 RTs: {result['sim_data']['rts'][:5]}")
    print(f"  First 5 Choices: {result['sim_data']['choices'][:5]}")


Shapes for std_ddm_1:
  sim_data:
    rts: (100,)
    choices: (100,)
  full_params: (100, 14)
  inference_conditions: (100, 46)

Shapes for coll_ddm_1:
  sim_data:
    rts: (100,)
    choices: (100,)
  full_params: (100, 16)
  inference_conditions: (100, 52)

Shapes for super_ddm_1:
  sim_data:
    rts: (100,)
    choices: (100,)
  full_params: (100, 19)
  inference_conditions: (100, 61)

Shapes for super_ddm_2:
  sim_data:
    rts: (100,)
    choices: (100,)
  full_params: (100, 22)
  inference_conditions: (100, 70)

Results for std_ddm_1:
  First 5 RTs: [1.132 0.849 0.703 0.429 0.493]
  First 5 Choices: [0. 1. 1. 1. 1.]

Results for coll_ddm_1:
  First 5 RTs: [1.245 1.264 1.533 2.619 0.943]
  First 5 Choices: [1. 0. 1. 1. 1.]

Results for super_ddm_1:
  First 5 RTs: [1.265 2.918 2.595 1.582 2.048]
  First 5 Choices: [0. 1. 1. 1. 1.]

Results for super_ddm_2:
  First 5 RTs: [2.188 1.451 2.175 3.57  1.544]
  First 5 Choices: [1. 1. 1. 1. 1.]


In [22]:
# Demonstrate clearing all variants
model_family.remove_all_variants()
print("\nVariants after clearing:", model_family.variant_names)


Variants after clearing: []
